# So sánh mô hình dự đoán thắng từ đội hình Agent
Notebook này chạy lại phép so sánh Logistic Regression, LightGBM và CatBoost. Đơn vị phân tích là một ván, với đội có mã viết tắt đứng trước theo thứ tự chữ cái làm nhãn đích. Chỉ dùng thông tin map, patch và đội hình Agent của hai bên.

Bộ test nằm hoàn toàn sau bộ train theo ngày. Đây là đánh giá khả năng dự đoán quan sát, không phải ước lượng tác động nhân quả của Agent.

In [ ]:
from pathlib import Path
import os, sys, subprocess
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
assert (root / 'data/processed/team_compositions.csv').exists(), 'Chạy make build trước'
subprocess.run([sys.executable, str(root / 'scripts/07_compare_predictors.py')], cwd=root, check=True)

In [ ]:
import json, pandas as pd
comparison = pd.read_csv(root / 'outputs/tables/predictor_comparison.csv')
split = json.loads((root / 'outputs/tables/predictor_split.json').read_text(encoding='utf-8'))
print({k: split[k] for k in ('train_first','train_last','test_first','test_last','n_train','n_test','positive_rate_train','positive_rate_test','raw_team1_win_rate','unseen_patch_test_games')})
display(comparison.style.format({'test_accuracy':'{:.3f}', 'test_roc_auc':'{:.3f}', 'test_log_loss':'{:.4f}', 'test_brier':'{:.4f}', 'delta_log_loss_vs_baseline':'{:+.4f}', 'delta_ci_low':'{:+.4f}', 'delta_ci_high':'{:+.4f}', 'fit_predict_seconds':'{:.1f}'}))

In [ ]:
by_map = pd.read_csv(root / 'outputs/tables/predictor_by_map.csv')
by_month = pd.read_csv(root / 'outputs/tables/predictor_by_month.csv')
display(by_map[by_map.model.isin(['LightGBM','CatBoost'])][['Map','model','n_games','roc_auc','delta_log_loss_vs_baseline']])
display(by_month[by_month.model.isin(['LightGBM','CatBoost'])][['month','model','n_games','roc_auc','delta_log_loss_vs_baseline']])

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(root / 'reports/figures/predictor_comparison.png')))

## Cách diễn giải
ROC AUC 0,5 tương đương xếp hạng ngẫu nhiên. Log loss và Brier càng nhỏ càng tốt. Cột `delta_log_loss_vs_baseline` dương nghĩa là hơn mốc tần suất train; `delta_ci_low` và `delta_ci_high` là khoảng tin cậy 95% từ 2.000 bootstrap theo MatchID. Nếu khoảng này chứa 0, chưa thể khẳng định mô hình hơn mốc. Team1 thắng bất thường trong dữ liệu gốc nên không được dùng làm đội đích. Phần lớn patch test chưa thấy trong train. Các bảng theo map và tháng cho biết kết quả có thể suy giảm ở đâu, không chứng minh Agent tạo ra chiến thắng.